## 计算nrrd文件的dice系数

In [2]:
import SimpleITK as sitk
import numpy as np

GT_path = "/workspace/data/Dice_label/通过DVF进行变换label的测试/01_inhale1_GT_label.nrrd"
predict_path = "/workspace/data/Dice_label/01/GT_label/01_inhale2_GT_label.nrrd"

# load data
seg1 = sitk.GetArrayFromImage(sitk.ReadImage(GT_path))
seg2 = sitk.GetArrayFromImage(sitk.ReadImage(predict_path))

# Compute Dice coefficient
dice = 2 * np.sum(seg1 * seg2) / (np.sum(seg1) + np.sum(seg2))
print(f"Dice coefficient: {dice}")

Dice coefficient: 0.7855072463768116


### 循环计算

In [2]:
import os
import SimpleITK as sitk
import numpy as np
import csv

def calculate_dice(label1_array, label2_array):
    """计算 Dice 系数"""
    intersection = np.sum(label1_array * label2_array)
    return (2 * intersection) / (np.sum(label1_array) + np.sum(label2_array))

def process_labels(gt_folder, predict_folder):
    """处理每个 GT 和 Predict 文件夹中的对应文件"""
    dice_results = []

    # 获取文件列表并排序（确保 GT 和 Predict 文件名一一对应）
    gt_files = sorted(os.listdir(gt_folder))
    predict_files = sorted(os.listdir(predict_folder))

    for gt_file, predict_file in zip(gt_files, predict_files):
        gt_path = os.path.join(gt_folder, gt_file)
        predict_path = os.path.join(predict_folder, predict_file)

        # 读取 .nrrd 文件
        gt_image = sitk.ReadImage(gt_path)
        predict_image = sitk.ReadImage(predict_path)

        # 转为 numpy 数组并确保是二值化数据
        gt_array = (sitk.GetArrayFromImage(gt_image) > 0).astype(np.uint8)
        predict_array = (sitk.GetArrayFromImage(predict_image) > 0).astype(np.uint8)

        # 计算 Dice 系数
        dice_score = calculate_dice(gt_array, predict_array)

        # 保存结果
        dice_results.append({
            "GT_file": gt_file,
            "Predict_file": predict_file,
            "Dice_coefficient": dice_score
        })

        print(f"Processed: {gt_file} vs {predict_file} -> Dice: {dice_score:.4f}")

    return dice_results

def save_results_to_csv(results, output_csv, mean_dice, std_dice):
    """将结果保存到 CSV 文件, 并附加平均值和标准差"""
    with open(output_csv, mode='w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=["GT_file", "Predict_file", "Dice_coefficient"])
        writer.writeheader()
        writer.writerows(results)

        # 在 CSV 的最后添加平均值和标准差
        writer.writerow({"GT_file": "Overall", "Predict_file": "Statistics", "Dice_coefficient": ""})
        writer.writerow({"GT_file": "Mean", "Predict_file": "", "Dice_coefficient": mean_dice})
        writer.writerow({"GT_file": "Std", "Predict_file": "", "Dice_coefficient": std_dice})

    print(f"Results saved to {output_csv}")

def calculate_stats(dice_scores):
    """计算 Dice 系数的平均值和标准差"""
    mean_dice = np.mean(dice_scores)
    std_dice = np.std(dice_scores)
    return mean_dice, std_dice

# 主程序
if __name__ == "__main__":
    base_folder = "/mnt/dataset/ouyang/dataset/Dice_label"  # 替换为你的数据集根目录
    output_csv = "/mnt/dataset/ouyang/dataset/Dice_label/dice_results.csv"

    all_results = []
    all_dice_scores = []

    # 遍历每个子文件夹
    for folder in sorted(os.listdir(base_folder)):
        gt_folder = os.path.join(base_folder, folder, "GT_label")
        predict_folder = os.path.join(base_folder, folder, "predict_label")

        if os.path.exists(gt_folder) and os.path.exists(predict_folder):
            print(f"Processing folder: {folder}")
            results = process_labels(gt_folder, predict_folder)
            all_results.extend(results)

            # 提取当前文件夹的 Dice 系数
            dice_scores = [result["Dice_coefficient"] for result in results]
            all_dice_scores.extend(dice_scores)

    # 计算总平均值和标准差
    mean_dice, std_dice = calculate_stats(all_dice_scores)

    # 保存结果到 CSV
    save_results_to_csv(all_results, output_csv, mean_dice, std_dice)

    # 计算总平均值和标准差
    print(f"\nOverall Dice Coefficient Mean: {mean_dice:.4f}")
    print(f"Overall Dice Coefficient Std: {std_dice:.4f}")

Processing folder: 01
Processed: 01_inhale1_GT_label.nrrd vs 01_inhale1_predict_label.nrrd -> Dice: 0.8737
Processed: 01_inhale2_GT_label.nrrd vs 01_inhale2_predict_label.nrrd -> Dice: 0.8669
Processed: 01_inhale3_GT_label.nrrd vs 01_inhale3_predict_label.nrrd -> Dice: 0.8363
Processing folder: 05
Processed: 05_inhale1_GT_label.nrrd vs 05_inhale1_predict_label.nrrd -> Dice: 0.8771
Processed: 05_inhale2_GT_label.nrrd vs 05_inhale2_predict_label.nrrd -> Dice: 0.8846
Processed: 05_inhale3_GT_label.nrrd vs 05_inhale3_predict_label.nrrd -> Dice: 0.8917
Processing folder: 06
Processed: 06_inhale1_GT_label.nrrd vs 06_inhale1_predict_label.nrrd -> Dice: 0.8378
Processed: 06_inhale2_GT_label.nrrd vs 06_inhale2_predict_label.nrrd -> Dice: 0.8544
Processed: 06_inhale3_GT_label.nrrd vs 06_inhale3_predict_label.nrrd -> Dice: 0.8383
Results saved to /mnt/dataset/ouyang/dataset/Dice_label/dice_results.csv

Overall Dice Coefficient Mean: 0.8623
Overall Dice Coefficient Std: 0.0201


### 显示方向变换

In [15]:
import SimpleITK as sitk

# Load the original .nrrd file
file_path = "/workspace/data/Dice_label/05_inhale3_GT - 副本.nrrd"  # Replace with your actual file path if different
output_path = "/workspace/data/Dice_label/05_inhale3_GT_flipped.nrrd"

# Read the image
image = sitk.ReadImage(file_path)

# Display original direction
original_direction = image.GetDirection()
print("Original Direction:", original_direction)

# Modify direction (flip Z-axis, for example)
new_direction = (1.0, 0.0, 0.0,
                 0.0, 1.0, 0.0,
                 0.0, 0.0, -1.0)  # Modify as needed based on your requirements

# Set the new direction to the image
image.SetDirection(new_direction)

# Save the modified image
sitk.WriteImage(image, output_path)

print("Flipped Direction:", image.GetDirection())
output_path

Original Direction: (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)
Flipped Direction: (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, -1.0)


'/workspace/data/Dice_label/05_inhale3_GT_flipped.nrrd'

#### 批量转换

In [3]:
import os
import SimpleITK as sitk

# Define paths
input_folder = "/workspace/data/Dice_label/01/GT_label"  # Replace with your dataset folder path
output_folder = "/workspace/data/Dice_label/Flipped"

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# List all files in the dataset folder
file_list = [f for f in os.listdir(input_folder) if f.endswith('.nrrd')]

# Transform all .nrrd files
for file_name in file_list:
    input_file_path = os.path.join(input_folder, file_name)
    output_file_path = os.path.join(output_folder, file_name)
    
    # Read the image
    image = sitk.ReadImage(input_file_path)
    
    # Modify direction (e.g., flip Z-axis)
    new_direction = (1.0, 0.0, 0.0,
                     0.0, 1.0, 0.0,
                     0.0, 0.0, -1.0)  # Modify as needed
    image.SetDirection(new_direction)
    
    # Save the modified image
    sitk.WriteImage(image, output_file_path)

output_folder

'/workspace/data/Dice_label/Flipped'

* 批量变换最终形

In [1]:
import os
import SimpleITK as sitk

# 主目录路径
base_dir = "/mnt/dataset/ouyang/dataset/Dice_label_flipped/GT_predict"  # 替换为您的主目录路径
output_base_dir = "/mnt/dataset/ouyang/dataset/Dice_label_flipped/GT_predict_flipped"  # 输出目录路径

# 创建输出主目录（如果不存在）
os.makedirs(output_base_dir, exist_ok=True)

# 遍历主目录中的所有文件和子目录
for root, dirs, files in os.walk(base_dir):
    for file_name in files:
        if file_name.endswith(".nrrd"):  # 只处理 .nrrd 文件
            input_file_path = os.path.join(root, file_name)
            
            # 计算输出文件的路径，保留目录结构
            relative_path = os.path.relpath(root, base_dir)
            output_dir = os.path.join(output_base_dir, relative_path)
            os.makedirs(output_dir, exist_ok=True)
            output_file_path = os.path.join(output_dir, file_name)
            
            # 读取 .nrrd 文件
            image = sitk.ReadImage(input_file_path)
            
            # 修改方向 (示例：翻转 Z 轴)
            new_direction = (1.0, 0.0, 0.0,
                             0.0, 1.0, 0.0,
                             0.0, 0.0, -1.0)  # 根据需要调整
            image.SetDirection(new_direction)
            
            # 保存修改后的 .nrrd 文件
            sitk.WriteImage(image, output_file_path)

print(f"所有文件已处理并保存到: {output_base_dir}")

所有文件已处理并保存到: /mnt/dataset/ouyang/dataset/Dice_label_flipped/GT_predict_flipped
